# ScanParts – Treino da IA de detecção de furos

Treina um **YOLO11n** (detecção) para achar furos em fotos de peças.

**Antes de começar:** menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4**.

O que este notebook faz:
1. Gera o **dataset sintético** (rótulo automático, custo zero).
2. (Opcional) Baixa um dataset do **Roboflow** e junta tudo.
3. (Opcional) Junta as **fotos do grupo** já anotadas.
4. Treina, avalia e salva tudo no **Google Drive** (a sessão do Colab pode cair).
5. Exporta o modelo em **ONNX** para levar ao Jetson.

Para gastar menos unidades: gere os dados e faça testes **sem GPU**, e só ligue a T4 para o treino.

## 1. GPU e Google Drive

In [ ]:
!nvidia-smi -L || echo "Sem GPU: tudo bem para gerar dados, mas troque para T4 antes de treinar"

from google.colab import drive
drive.mount('/content/drive')

PASTA = '/content/drive/MyDrive/scanparts_ia'   # tudo que importa fica salvo aqui
import os
os.makedirs(PASTA, exist_ok=True)
print('Salvando em', PASTA)

## 2. Código do projeto

Baixa o repositório. Se ele for **privado**, crie um token no GitHub (permissão de leitura) e salve nos
**Secrets** do Colab (ícone de chave 🔑 na barra lateral) com o nome `GITHUB_TOKEN`.

In [ ]:
REPO = 'github.com/Gustavo-dev04/Baja.git'
BRANCH = 'claude/wizardly-dirac-mnqwsx'   # troque para 'main' quando o código estiver lá

try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    url = f'https://{token}@{REPO}'
except Exception:
    url = f'https://{REPO}'

!rm -rf /content/scanparts
!git clone -q --depth 1 -b {BRANCH} {url} /content/scanparts && echo "Código baixado"
!ls /content/scanparts/ia

## 3. Bibliotecas

In [ ]:
!pip -q install ultralytics opencv-python-headless
import ultralytics
ultralytics.checks()

## 4. Dataset sintético

Gera as imagens uma vez e guarda um `.zip` no Drive. Nas próximas vezes, só descompacta (bem mais rápido).

In [ ]:
N_TREINO, N_VAL = 3000, 500
ZIP = f'{PASTA}/sintetico_{N_TREINO}_{N_VAL}.zip'

if os.path.exists(ZIP):
    !unzip -q -o "{ZIP}" -d /content/dados && echo "Dataset sintético restaurado do Drive"
else:
    !python /content/scanparts/ia/gerar_sintetico.py --saida /content/dados/sintetico \
        --treino {N_TREINO} --val {N_VAL} --previa 12
    !cd /content/dados && zip -q -r "{ZIP}" sintetico && echo "Zip salvo no Drive"

In [ ]:
# Confere algumas imagens com os rótulos desenhados
import glob
from IPython.display import Image as Img, display
for f in sorted(glob.glob('/content/dados/sintetico/previa/*.jpg'))[:6]:
    display(Img(filename=f, width=320))

## 5. (Opcional) Dataset do Roboflow

1. Crie uma conta grátis em [roboflow.com](https://roboflow.com) e copie sua **API key**.
2. Salve nos Secrets do Colab como `ROBOFLOW_API_KEY`.
3. Abra o dataset no Roboflow Universe, clique em **Download → YOLOv11** e copie o *workspace*, o *project* e a *version*.
4. Em `IDS_FURO`, coloque o número das classes que são furo (veja no `data.yaml` baixado).

Sugestão: *drill-hole-detection* (260 imagens, licença CC BY 4.0). **Confira a licença de cada dataset.**

In [ ]:
USAR_ROBOFLOW = False
WORKSPACE, PROJETO, VERSAO = 'corbinian-rehm-master-thesis', 'drill-hole-detection', 1
IDS_FURO = '0'

if USAR_ROBOFLOW:
    !pip -q install roboflow
    from google.colab import userdata
    from roboflow import Roboflow
    rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
    ds = rf.workspace(WORKSPACE).project(PROJETO).version(VERSAO).download('yolov11', location='/content/dados/roboflow')
    !cat /content/dados/roboflow/data.yaml

## 6. (Opcional) Fotos do grupo

Anote as fotos no **CVAT** ou no **Label Studio**, exporte em formato **YOLO** e suba o `.zip`
para `scanparts_ia/fotos_grupo.zip` no Drive.

In [ ]:
FOTOS = f'{PASTA}/fotos_grupo.zip'
USAR_FOTOS = os.path.exists(FOTOS)
if USAR_FOTOS:
    !unzip -q -o "{FOTOS}" -d /content/dados/fotos_grupo && echo "Fotos do grupo carregadas"
else:
    print('Sem fotos do grupo por enquanto')

## 7. Junta tudo num dataset só

In [ ]:
fontes = ['/content/dados/sintetico']
if USAR_ROBOFLOW:
    fontes.append(f'/content/dados/roboflow:{IDS_FURO}')
if USAR_FOTOS:
    fontes.append('/content/dados/fotos_grupo')

!rm -rf /content/dados/final
!python /content/scanparts/ia/juntar_datasets.py --destino /content/dados/final {' '.join(fontes)}
DATA = '/content/dados/final/data.yaml'

## 8. Treino

- `yolo11n` é o menor modelo: treina rápido na T4 e roda bem no Jetson.
- Os resultados ficam em `scanparts_ia/treinos/` no Drive, com checkpoint a cada 5 épocas.
- Se a sessão cair **no meio** do treino, rode a célula **8b**. Se o treino terminou, pule a 8b.

In [ ]:
from ultralytics import YOLO

NOME = 'furos_v3'   # troque a cada treino novo (v3, v4...) para não sobrescrever o anterior
model = YOLO('yolo11n.pt')
model.train(
    data=DATA,
    epochs=80,
    imgsz=640,
    batch=32,
    patience=20,          # para sozinho se parar de melhorar
    project=f'{PASTA}/treinos',
    name=NOME,
    exist_ok=True,
    save_period=5,
    workers=2,
    fliplr=0.5, flipud=0.5, degrees=15,
)

In [ ]:
# 8b. Continuar um treino interrompido — SÓ use se a sessão caiu no meio do treino
import torch
from ultralytics import YOLO
NOME = globals().get('NOME', 'furos_v3')
ultimo = f'{PASTA}/treinos/{NOME}/weights/last.pt'
ck = torch.load(ultimo, map_location='cpu', weights_only=False)
if ck.get('optimizer') is None:
    print('Esse treino já terminou, não há nada para continuar. Pule para a célula 9.')
else:
    YOLO(ultimo).train(resume=True)

## 9. Resultado

In [ ]:
from IPython.display import Image as Img, display
pasta_treino = f'{PASTA}/treinos/{NOME}'
display(Img(filename=f'{pasta_treino}/results.png', width=900))

best = YOLO(f'{pasta_treino}/weights/best.pt')
m = best.val(data=DATA)
print(f"Precisão: {m.box.mp:.3f} | Recall (furos achados): {m.box.mr:.3f} | mAP50: {m.box.map50:.3f}")

In [ ]:
# Exemplos de detecção nas imagens de validação
import glob
imgs = sorted(glob.glob('/content/dados/final/images/val/*'))[:6]
for r in best.predict(imgs, conf=0.4, save=True, project='/content/pred', name='val', exist_ok=True):
    pass
for f in sorted(glob.glob('/content/pred/val/*'))[:6]:
    display(Img(filename=f, width=320))

## 10. Teste com fotos reais (sobe do computador)

In [ ]:
from google.colab import files
enviadas = files.upload()
for r in best.predict(list(enviadas), conf=0.4, save=True, project='/content/pred', name='reais', exist_ok=True):
    print(r.path, '->', len(r.boxes), 'furos')
for f in sorted(glob.glob('/content/pred/reais/*')):
    display(Img(filename=f, width=480))

## 11. Exportar para o Jetson

Exporta em **ONNX** (salvo no Drive). O arquivo TensorRT (`.engine`) **tem que ser gerado no próprio Jetson**,
porque ele é específico da placa:

```bash
yolo export model=best.pt format=engine half=True imgsz=640
```

In [ ]:
caminho = best.export(format='onnx', imgsz=640)
!cp "{caminho}" "{PASTA}/" && cp "{pasta_treino}/weights/best.pt" "{PASTA}/"
print('Salvos no Drive: best.pt e', os.path.basename(caminho))